In [39]:
import json
from typing import Literal

from pydantic import BaseModel
from minsearch import Index
from anthropic import Anthropic
from gitsource import GithubRepositoryDataReader, chunk_documents

In [40]:
client = Anthropic()

INSTRUCTIONS = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT.
"""

PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

In [10]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool
    confidence: float
    confidence_explanation: str
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"]
    followup_questions: list[str]

In [41]:
class RAG:
    def __init__(
        self,
        index: Index,
        llm_client: Anthropic,
        output_type: type[BaseModel],
        instructions: str = INSTRUCTIONS,
    ):
        self.index = index
        self.llm_client = llm_client
        self.output_type = output_type
        self.instructions = instructions

    def search(self, query: str, top_k: int = 5):
        return self.index.search(query, num_results=top_k)

    def build_prompt(self, question: str, search_results: list):
        context = json.dumps(search_results, indent=2)
        return PROMPT_TEMPLATE.format(
            question=question,
            context=context
        )

    def llm(
        self,
        user_prompt: str,
        model: str = "claude-opus-5",
        max_tokens: int = 2056,
    ):
        with self.llm_client.messages.stream(
            model=model,
            max_tokens=max_tokens,
            messages=[
                {"role": "assistant", "content": self.instructions or "You are a helpful course assistant."},
                {"role": "user", "content": user_prompt}
            ],
            thinking={"type": "adaptive", "display": "summarized"},
            output_format=self.output_type,
        ) as stream:
            response = stream.get_final_message()
            return response            

    def invoke(self, query: str):
        search_results = self.search(query=query)
        user_prompt = self.build_prompt(question=query, search_results=search_results)
        response = self.llm(user_prompt=user_prompt)
        
        output = None
        for block in response.content:
            if block.type == "text":
                output = block.parsed_output
        return output

In [14]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

In [15]:
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [16]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

Indexed 382 chunks from 95 documents


In [31]:
rag = RAG(
    index=index,
    llm_client=client,
    output_type=RAGResponse,
)
response = rag.invoke(query="LLM as a judge")

In [42]:
print(f"Found answer: {response.found_answer}")  # type: ignore
print(f"Confidence: {response.confidence}")  # type: ignore
print(f"Answer type: {response.answer_type}")  # type: ignore
print(f"Follow-up questions: {response.followup_questions}")  # type: ignore

Found answer: True
Confidence: 0.85
Answer type: explanation
Follow-up questions: ['Do you want help writing the judge prompt criteria for a specific use case?', 'Would you like to see how to switch to a different evaluator LLM (not OpenAI)?', 'Are you interested in using the multi-class template instead of binary classification?', 'Do you want to know how to integrate the judge into regression testing after a prompt change?']
